# VQA-RAD Pilot — Zero-Shot Closed-Ended Medical VQA

**Model:** Qwen2.5-VL-3B-Instruct (zero-shot, no fine-tuning, no few-shot examples)
**Sample:** 100 closed-ended (yes/no) question-answer pairs, balanced close to 50/50
**Purpose:** Small-scale pilot to demonstrate full analysis capability before the full 3-model study.

Run all cells top to bottom on a Colab or Kaggle GPU runtime (T4 or better).

In [1]:
!pip install -q transformers accelerate qwen-vl-utils pillow pandas scikit-learn torch datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 39.4 MB/s eta 0:00:00


## Step 1 — Load VQA-RAD and filter to the closed-ended (yes/no) subset

VQA-RAD's own leaderboard defines "closed-ended" as the binary yes/no question subset,
so filtering by `answer in {"yes", "no"}` matches the dataset's own convention exactly.

In [2]:
from datasets import load_dataset
import pandas as pd

vqa_rad = load_dataset("flaviagiammarino/vqa-rad")
full_df = pd.concat(
    [vqa_rad["train"].to_pandas(), vqa_rad["test"].to_pandas()], ignore_index=True
)

full_df["answer_norm"] = full_df["answer"].astype(str).str.strip().str.lower()
closed_df = full_df[full_df["answer_norm"].isin(["yes", "no"])].reset_index(drop=True)
print(f"Closed-ended (yes/no) pairs available: {len(closed_df)}")
print(closed_df["answer_norm"].value_counts())

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:121: UserWarning: 
Access to the secret `HF_TOKEN` has not been granted on this notebook.
You will not be requested again.
Please restart the session if you want to be prompted again.
  warnings.warn(


README.md:   0%|          | 0.00/3.91k [00:00<?, ?B/s]

data/train-00000-of-00001-eb8844602202be(…): reconstructing file:   0%|          |  0.00B / 24.2MB            

data/train-00000-of-00001-eb8844602202be(…): downloading bytes:           |  0.00B            

data/test-00000-of-00001-e5bc3d208bb4dee(…): reconstructing file:   0%|          |  0.00B / 10.3MB            

data/test-00000-of-00001-e5bc3d208bb4dee(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/1793 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/451 [00:00<?, ? examples/s]

Closed-ended (yes/no) pairs available: 1191
answer_norm
no     606
yes    585
Name: count, dtype: int64


## Step 2 — Sample 100 pairs, balanced close to 50/50 yes/no

In [3]:
SEED = 42
N_TOTAL = 100
N_PER_ANSWER = N_TOTAL // 2

sampled = (
    closed_df.groupby("answer_norm", group_keys=False)
    .apply(lambda g: g.sample(n=min(N_PER_ANSWER, len(g)), random_state=SEED))
    .reset_index(drop=True)
)
print(f"Total sampled: {len(sampled)}")
sampled[["question", "answer_norm"]].to_csv("vqa_rad_pilot_sample.csv", index=False)
sampled[["question", "answer_norm"]].head(10)

Total sampled: 100


/tmp/ipykernel_742/345975756.py:7: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(n=min(N_PER_ANSWER, len(g)), random_state=SEED))


,question,answer_norm
0,is this a normal image?,no
1,is there evidence of herniation of the small b...,no
2,was the patient positioned inappropriately?,no
3,does the left temporal lobe appear normal?,no
4,is this an anterior-posterior image,no
5,are the ventricles an abnormal size?,no
6,is a pleural effusion present?,no
7,are the lungs increased in size?,no
8,is the cardiac silhouette enlarged?,no
9,is there cardiomegaly?,no


## Step 3 — Save the sampled images to disk

The dataset stores images as embedded PIL objects rather than file paths, so we save them once to a local folder for reliable loading during inference.

In [5]:
from pathlib import Path
from PIL import Image
import io

IMG_DIR = Path("vqa_rad_pilot_images")
IMG_DIR.mkdir(exist_ok=True)

def to_pil(img_field):
    # to_pandas() leaves HF Image columns as dicts {'bytes': ..., 'path': ...}
    # instead of decoded PIL Images, so decode manually here.
    if isinstance(img_field, dict):
        return Image.open(io.BytesIO(img_field["bytes"]))
    return img_field  # already a PIL Image

image_paths = []
for i, row in sampled.iterrows():
    path = IMG_DIR / f"{i}.jpg"
    to_pil(row["image"]).convert("RGB").save(path)
    image_paths.append(str(path))

sampled["image_path"] = image_paths

## Step 4 — Load Qwen2.5-VL-3B-Instruct

In [6]:
import torch
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor

MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map="auto"
)
processor = AutoProcessor.from_pretrained(MODEL_ID)

config.json:   0%|          | 0.00/1.37k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/65.4k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

...

chat_template.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/5.70k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

## Step 5 — Zero-shot prompt and inference loop

In [7]:
from qwen_vl_utils import process_vision_info

def build_prompt(question):
    return (
        "You are a radiology assistant. Look at this medical image and answer the following "
        "question with ONLY the single word 'Yes' or 'No'.\n\n"
        f"Question: {question}"
    )

def predict(image_path, question):
    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": image_path},
            {"type": "text", "text": build_prompt(question)},
        ],
    }]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text], images=image_inputs, videos=video_inputs, padding=True, return_tensors="pt"
    ).to(model.device)
    with torch.no_grad():
        generated = model.generate(**inputs, max_new_tokens=16)
    trimmed = [out[len(inp):] for inp, out in zip(inputs.input_ids, generated)]
    return processor.batch_decode(trimmed, skip_special_tokens=True)[0].strip()

def parse_yes_no(raw_output):
    lowered = raw_output.lower()
    idx_yes = lowered.find("yes")
    idx_no = lowered.find("no")
    if idx_yes == -1 and idx_no == -1:
        return "unparsed"
    if idx_yes == -1:
        return "no"
    if idx_no == -1:
        return "yes"
    return "yes" if idx_yes < idx_no else "no"


In [8]:
results = []
for i, row in sampled.iterrows():
    raw = predict(row["image_path"], row["question"])
    pred = parse_yes_no(raw)
    results.append({
        "question": row["question"],
        "true_answer": row["answer_norm"],
        "predicted_answer": pred,
        "raw_output": raw,
    })
    if (i + 1) % 10 == 0:
        print(f"{i + 1}/{len(sampled)} done")

results_df = pd.DataFrame(results)
results_df.to_csv("vqa_rad_pilot_results.csv", index=False)
results_df.head()

10/100 done
20/100 done
30/100 done
40/100 done
50/100 done
60/100 done
70/100 done
80/100 done
90/100 done
100/100 done


,question,true_answer,predicted_answer,raw_output
0,is this a normal image?,no,no,No
1,is there evidence of herniation of the small b...,no,yes,Yes
2,was the patient positioned inappropriately?,no,no,No
3,does the left temporal lobe appear normal?,no,yes,Yes
4,is this an anterior-posterior image,no,yes,Yes


## Step 6 — Score the results

In [9]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

valid = results_df[results_df["predicted_answer"] != "unparsed"]
print(f"Parsed cleanly: {len(valid)}/{len(results_df)}")

acc = accuracy_score(valid["true_answer"], valid["predicted_answer"])
print(f"Overall closed-ended accuracy: {acc:.3f}\n")

print(classification_report(valid["true_answer"], valid["predicted_answer"], zero_division=0))

cm = confusion_matrix(valid["true_answer"], valid["predicted_answer"], labels=["yes", "no"])
cm_df = pd.DataFrame(cm, index=["true_yes", "true_no"], columns=["pred_yes", "pred_no"])
cm_df.to_csv("vqa_rad_pilot_confusion_matrix.csv")
cm_df

Parsed cleanly: 100/100
Overall closed-ended accuracy: 0.680

              precision    recall  f1-score   support

          no       0.70      0.64      0.67        50
         yes       0.67      0.72      0.69        50

    accuracy                           0.68       100
   macro avg       0.68      0.68      0.68       100
weighted avg       0.68      0.68      0.68       100



,pred_yes,pred_no
true_yes,36,14
true_no,18,32


## Step 7 — Pull out the misclassified cases for your qualitative write-up

In [10]:
wrong = results_df[results_df["predicted_answer"] != results_df["true_answer"]]
print(f"{len(wrong)} misclassified out of {len(results_df)}")
wrong[["question", "true_answer", "predicted_answer", "raw_output"]]

32 misclassified out of 100


,question,true_answer,predicted_answer,raw_output
1,is there evidence of herniation of the small b...,no,yes,Yes
3,does the left temporal lobe appear normal?,no,yes,Yes
4,is this an anterior-posterior image,no,yes,Yes
11,are the hepatic lesions ring enhancing?,no,yes,Yes
12,is mass effect present?,no,yes,Yes
16,is there evidence of free peritoneal fluid?,no,yes,Yes
19,is the fat surrounding the pancreas normal?,no,yes,Yes
20,are there any pulmonary findings?,no,yes,Yes
24,is a cystic cavity present in the left kidney ...,no,yes,Yes
27,can a pulmonary mass be appreciated?,no,yes,Yes
